<a href="https://colab.research.google.com/github/dimalgeorge3/SUGNT-Framework/blob/main/MiniProject_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

# Uninstall all related packages first to ensure a clean slate
!pip uninstall -y torch torchaudio torchvision

# Install a stable and compatible set of versions for PyTorch with CUDA 12.1
# Using torch==2.2.0, torchvision==0.17.0, and torchaudio==2.2.0 as they are commonly compatible.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 763.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 26.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 40.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 116.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

Confidence based Early Exit

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset

# 1. THE 12-LAYER EARLY EXIT ARCHITECTURE
class EarlyExitBERT12(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=2):
        super().__init__()
        # Load pre-trained 12-layer BERT
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size # 768 for BERT-base

        # We attach exits at strategic depths:
        # Layer 3 (Fast), Layer 7 (Medium), Layer 12 (Full)
        self.exit_1 = nn.Linear(hidden_size, num_classes)
        self.exit_2 = nn.Linear(hidden_size, num_classes)
        self.exit_final = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, threshold=None):
        # Forward pass through BERT backbone
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        all_hidden_states = outputs.hidden_states

        # --- EXIT 1: AFTER LAYER 3 ---
        # all_hidden_states[3] corresponds to the output of the 3rd Encoder block
        h3 = all_hidden_states[3][:, 0, :] # [CLS] token
        logits1 = self.exit_1(h3)

        if not self.training and threshold:
            if F.softmax(logits1, dim=-1).max().item() >= threshold:
                return logits1, 3 # Exit early at Layer 3

        # --- EXIT 2: AFTER LAYER 7 ---
        h7 = all_hidden_states[7][:, 0, :]
        logits2 = self.exit_2(h7)

        if not self.training and threshold:
            if F.softmax(logits2, dim=-1).max().item() >= threshold:
                return logits2, 7 # Exit early at Layer 7

        # --- FINAL EXIT: AFTER LAYER 12 ---
        h12 = all_hidden_states[12][:, 0, :]
        logits_f = self.exit_final(h12)

        return (logits1, logits2, logits_f) if self.training else (logits_f, 12)

# 2. SETUP & DATA LOADING
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = EarlyExitBERT12().to(device)

# Load 1000 samples from Amazon Polarity
dataset = load_dataset("amazon_polarity", split="train", trust_remote_code=True).shuffle(seed=42).select(range(500))

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# 3. FINE-TUNING LOOP
print("Fine-tuning 12-layer Early Exit BERT on 1000 Amazon Reviews...")
model.train()
for epoch in range(5):
    for i, item in enumerate(dataset):
        optimizer.zero_grad()
        inputs = tokenizer(item['content'], return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        label = torch.tensor([item['label']]).to(device)

        # Training returns all three exit logits
        l1, l2, lf = model(inputs['input_ids'], inputs['attention_mask'])

        # Weighted loss (Final exit has slightly more weight)
        loss = (0.2 * criterion(l1, label)) + (0.3 * criterion(l2, label)) + (0.5 * criterion(lf, label))
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} Complete.")

# 4. INTERACTIVE TESTING
def analyze_review(text, threshold=0.65):
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
        logits, exit_layer = model(inputs['input_ids'], inputs['attention_mask'], threshold=threshold)

        prob = F.softmax(logits, dim=-1)
        conf, pred = torch.max(prob, dim=-1)
        sentiment = "POSITIVE ✅" if pred.item() == 1 else "NEGATIVE ❌"

        print(f"\nReview: {text[:80]}...")
        print(f"Verdict: {sentiment} | Confidence: {conf.item():.2%} | Exit Point: Layer {exit_layer}")

# Run Examples
analyze_review("This is an absolutely perfect product. Highly recommended!")
analyze_review("Waste of money. It arrived broken and the customer support is terrible.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'amazon_polarity' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it t

Fine-tuning 12-layer Early Exit BERT on 1000 Amazon Reviews...
Epoch 1 Complete.
Epoch 2 Complete.
Epoch 3 Complete.
Epoch 4 Complete.
Epoch 5 Complete.

Review: This is an absolutely perfect product. Highly recommended!...
Verdict: POSITIVE ✅ | Confidence: 99.14% | Exit Point: Layer 3

Review: Waste of money. It arrived broken and the customer support is terrible....
Verdict: NEGATIVE ❌ | Confidence: 99.83% | Exit Point: Layer 3


BERT based with M C dropout (2 exit)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset

class MCUncertaintyBERT(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=2):
        super().__init__()
        # Load the 12-layer BERT backbone
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # MC Dropout requires the dropout layer to remain active
        self.dropout = nn.Dropout(p=0.1)

        # Two Exit Points: Early (Layer 4) and Final (Layer 12)
        self.exit_mid = nn.Linear(hidden_size, num_classes)
        self.exit_final = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, num_passes=10, var_threshold=0.005):
        # Forward pass to get all hidden states
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # --- TRAINING MODE ---
        if self.training:
            h4 = outputs.hidden_states[4][:, 0, :]
            h12 = outputs.hidden_states[12][:, 0, :]
            return self.exit_mid(h4), self.exit_final(h12)

        # --- INFERENCE MODE (MC DROPOUT) ---
        h4 = outputs.hidden_states[4][:, 0, :]

        # Step 1: Force Dropout to stay ON during inference
        self.dropout.train()

        stochastic_logits = []
        for _ in range(num_passes):
            # Pass through dropout then the middle exit head
            out = self.exit_mid(self.dropout(h4))
            stochastic_logits.append(F.softmax(out, dim=-1))

        # Step 2: Calculate Variance (Uncertainty)
        stacked_probs = torch.stack(stochastic_logits) # [Passes, Batch, Classes]
        mean_probs = stacked_probs.mean(dim=0)
        variance = stacked_probs.var(dim=0).mean().item() # Mean variance across classes

        # Step 3: Check Uncertainty Threshold
        if variance < var_threshold:
            return mean_probs, 4, variance # EXIT EARLY

        # Step 4: If uncertain, compute the final 12th layer
        h12 = outputs.hidden_states[12][:, 0, :]
        final_logits = self.exit_final(h12)
        return F.softmax(final_logits, dim=-1), 12, variance

# --- 3. Execution & Testing ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = MCUncertaintyBERT().to(device)

# Load Amazon Polarity Sample
dataset = load_dataset("amazon_polarity", split="train", trust_remote_code=True).shuffle().select(range(500))

def test_review(text, threshold=0.002):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    with torch.no_grad():
        probs, layer, uncertainty = model(inputs['input_ids'], inputs['attention_mask'],
                                          num_passes=10, var_threshold=threshold)

    pred = torch.argmax(probs).item()
    status = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"\nReview: {text[:70]}...")
    print(f"Result: {status} | Exit Layer: {layer} | Uncertainty: {uncertainty:.6f}")

# Example Test Cases
test_review("This is the best purchase I have ever made, absolutely perfect!") # Likely Exit 4
test_review("The product was okay, but I'm not sure if the price is justified.") # Likely Exit 12

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'amazon_polarity' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it t


Review: This is the best purchase I have ever made, absolutely perfect!...
Result: POSITIVE | Exit Layer: 4 | Uncertainty: 0.000824

Review: The product was okay, but I'm not sure if the price is justified....
Result: POSITIVE | Exit Layer: 4 | Uncertainty: 0.001021


MC dropout based Early Exit (3)

---



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm

# --- 1. Model Definition with 3 Exits ---
class MCUncertaintyBERT(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(p=0.1)

        # Three Exit Points
        self.exit_4 = nn.Linear(hidden_size, num_classes)
        self.exit_8 = nn.Linear(hidden_size, num_classes)
        self.exit_final = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, num_passes=10, var_threshold=0.002):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        if self.training:
            return (self.exit_4(outputs.hidden_states[4][:, 0, :]),
                    self.exit_8(outputs.hidden_states[8][:, 0, :]),
                    self.exit_final(outputs.hidden_states[12][:, 0, :]))

        # INFERENCE MODE
        self.dropout.train()

        # Exit 1 (Layer 4)
        h4 = outputs.hidden_states[4][:, 0, :]
        probs_4, var_4 = self._get_mc_preds(h4, self.exit_4, num_passes)
        if var_4 < var_threshold:
            return probs_4, 4, var_4

        # Exit 2 (Layer 8)
        h8 = outputs.hidden_states[8][:, 0, :]
        probs_8, var_8 = self._get_mc_preds(h8, self.exit_8, num_passes)
        if var_8 < var_threshold:
            return probs_8, 8, var_8

        # Final Exit (Layer 12)
        h12 = outputs.hidden_states[12][:, 0, :]
        final_logits = self.exit_final(h12)
        return F.softmax(final_logits, dim=-1), 12, var_8

    def _get_mc_preds(self, hidden_state, exit_head, num_passes):
        stochastic_logits = []
        for _ in range(num_passes):
            out = exit_head(self.dropout(hidden_state))
            stochastic_logits.append(F.softmax(out, dim=-1))
        stacked = torch.stack(stochastic_logits)
        return stacked.mean(dim=0), stacked.var(dim=0).mean().item()

# --- 2. Setup & Evaluation ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = MCUncertaintyBERT().to(device)

# Using a subset of the test split for the Evaluation Matrix
test_dataset = load_dataset("amazon_polarity", split="test", trust_remote_code=True).shuffle(seed=42).select(range(1000))

def run_evaluation(threshold=0.002):
    model.eval()
    correct, total = 0, 0
    exits = {4: 0, 8: 0, 12: 0}

    print(f"\n[System] Evaluating model performance (Threshold: {threshold})...")
    for item in tqdm(test_dataset):
        inputs = tokenizer(item['content'], return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            probs, layer, _ = model(inputs['input_ids'], inputs['attention_mask'], var_threshold=threshold)

        if torch.argmax(probs).item() == item['label']:
            correct += 1
        total += 1
        exits[layer] += 1

    print("\n" + "="*30)
    print(f"EVALUATION MATRIX (N={total})")
    print("-" * 30)
    print(f"Total Accuracy: {correct/total:.2%}")
    print(f"Layer 4 Exits:  {exits[4]} samples")
    print(f"Layer 8 Exits:  {exits[8]} samples")
    print(f"Layer 12 Exits: {exits[12]} samples")
    print("="*30)

# --- 3. Simple Interactive Loop ---
def start_interactive_session(threshold=0.002):
    print("\nReady! Enter your review below (or type 'quit' to stop).")
    while True:
        text = input("\nReview > ")
        if text.lower() in ['quit', 'exit', 'q']: break

        model.eval()
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

        with torch.no_grad():
            probs, layer, var = model(inputs['input_ids'], inputs['attention_mask'], var_threshold=threshold)

        sentiment = "POSITIVE" if torch.argmax(probs).item() == 1 else "NEGATIVE"
        color = "\033[92m" if sentiment == "POSITIVE" else "\033[91m"
        reset = "\033[0m"

        print(f"Result: {color}{sentiment}{reset} | Exit: Layer {layer} | Uncertainty: {var:.6f}")

if __name__ == "__main__":
    run_evaluation()
    start_interactive_session()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import os # Import os for file system checks

# --- 1. MODEL ARCHITECTURE ---
class MCUncertaintyBERT(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(p=0.1)

        # Three Exit Heads: Randomly initialized, need fine-tuning
        self.exit_4 = nn.Linear(hidden_size, num_classes)
        self.exit_8 = nn.Linear(hidden_size, num_classes)
        self.exit_final = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, num_passes=10, var_threshold=0.002):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        # --- TRAIN MODE ---
        if self.training:
            # We use the [CLS] token (index 0) from the hidden states
            h4 = outputs.hidden_states[4][:, 0, :]
            h8 = outputs.hidden_states[8][:, 0, :]
            h12 = outputs.hidden_states[12][:, 0, :]
            return self.exit_4(h4), self.exit_8(h8), self.exit_final(h12)

        # --- INFERENCE MODE (MC DROPOUT) ---
        # Step 1: Force dropout to remain active
        self.dropout.train()

        # Check Exit 1 (Layer 4)
        h4 = outputs.hidden_states[4][:, 0, :]
        probs_4, var_4 = self._get_mc_preds(h4, self.exit_4, num_passes)
        if var_4 < var_threshold:
            return probs_4, 4, var_4

        # Check Exit 2 (Layer 8)
        h8 = outputs.hidden_states[8][:, 0, :]
        probs_8, var_8 = self._get_mc_preds(h8, self.exit_8, num_passes)
        if var_8 < var_threshold:
            return probs_8, 8, var_8

        # Final Exit (Layer 12)
        h12 = outputs.hidden_states[12][:, 0, :]
        final_logits = self.exit_final(h12)
        return F.softmax(final_logits, dim=-1), 12, var_8 # Returning var_8 as a proxy

    def _get_mc_preds(self, hidden_state, exit_head, num_passes):
        stochastic_logits = []
        for _ in range(num_passes):
            # Applying dropout to the hidden state multiple times
            out = exit_head(self.dropout(hidden_state))
            stochastic_logits.append(F.softmax(out, dim=-1))

        stacked = torch.stack(stochastic_logits) # [Passes, Batch, Classes]
        mean_probs = stacked.mean(dim=0)
        # Variance across the stochastic passes
        variance = stacked.var(dim=0).mean().item()
        return mean_probs, variance

# --- 2. DATA PREPARATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Dataset sizes - adjust range() to increase/decrease
print("Loading datasets...")
train_raw = load_dataset("amazon_polarity", split="train").shuffle(seed=42).select(range(2000))
test_raw = load_dataset("amazon_polarity", split="test").shuffle(seed=42).select(range(500))

def tokenize_fn(batch):
    return tokenizer(batch['content'], padding='max_length', truncation=True, max_length=128)

train_ds = train_raw.map(tokenize_fn, batched=True)
test_ds = test_raw.map(tokenize_fn, batched=True)
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1) # Set to 1 for clean exit tracking

# --- 3. THE TRAINING ENGINE ---
model = MCUncertaintyBERT().to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

def train_model(epochs=1):
    print(f"\n[Step 1/3] Training {epochs} Epoch(s)...")
    model.train()
    for epoch in range(epochs):
        loop = tqdm(train_loader, leave=True)
        for batch in loop:
            optimizer.zero_grad()
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            # Get predictions for all three exits
            out4, out8, out12 = model(ids, mask)

            # Loss is sum of all three exits
            loss = criterion(out4, labels) + criterion(out8, labels) + criterion(out12, labels)

            loss.backward()
            optimizer.step()
            loop.set_description(f"Epoch {epoch}")
            loop.set_postfix(loss=loss.item())

# --- 4. PERFORMANCE EVALUATION ---
def run_evaluation(threshold=0.002):
    model.eval()
    correct, total = 0, 0
    exits = {4: 0, 8: 0, 12: 0}

    print("\n[Step 2/3] Generating Performance Matrix...")
    for batch in tqdm(test_loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        with torch.no_grad():
            # Evaluation uses the fixed threshold
            probs, layer, _ = model(ids, mask, var_threshold=threshold)
            pred = torch.argmax(probs, dim=-1)

            if pred.item() == labels.item():
                correct += 1
            total += 1
            exits[layer] += 1

    print("\n" + "="*40)
    print(f"FINAL PERFORMANCE MATRIX")
    print("-" * 40)
    print(f"Overall Accuracy: {correct/total:.2%}")
    print(f"Layer 4 Exits:  {exits[4]} ({(exits[4]/total)*100:.1f}%)")
    print(f"Layer 8 Exits:  {exits[8]} ({(exits[8]/total)*100:.1f}%)")
    print(f"Layer 12 Exits: {exits[12]} ({(exits[12]/total)*100:.1f}%)")
    print("="*40)

# --- 5. INTERACTIVE TESTING ---
def interactive_tester(threshold=0.002):
    print("\n[Step 3/3] Ready! Enter a review (type 'exit' to stop).")
    while True:
        text = input("\nReview: ")
        if text.lower() in ['q', 'quit', 'exit']: break

        model.eval()
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

        with torch.no_grad():
            probs, layer, var = model(inputs['input_ids'], inputs['attention_mask'], var_threshold=threshold)

        sentiment = "POSITIVE" if torch.argmax(probs).item() == 1 else "NEGATIVE"
        print(f">> Result: {sentiment} | Exit Layer: {layer} | Variance: {var:.6f}")

if __name__ == "__main__":
    MODEL_PATH = 'mc_uncertainty_bert.pth'

    if os.path.exists(MODEL_PATH):
        print(f"\nLoading trained model from {MODEL_PATH}...")
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print("Model loaded successfully.")
    else:
        train_model(epochs=1) # Start training
        print(f"\nSaving trained model to {MODEL_PATH}...")
        torch.save(model.state_dict(), MODEL_PATH)
        print("Model saved.")

    run_evaluation()      # Evaluate
    interactive_tester()  # Test


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


[Step 1/3] Training 1 Epoch(s)...


Epoch 0: 100%|██████████| 125/125 [42:03<00:00, 20.19s/it, loss=1.33]



Saving trained model to mc_uncertainty_bert.pth...
Model saved.

[Step 2/3] Generating Performance Matrix...


100%|██████████| 500/500 [04:16<00:00,  1.95it/s]



FINAL PERFORMANCE MATRIX
----------------------------------------
Overall Accuracy: 81.20%
Layer 4 Exits:  223 (44.6%)
Layer 8 Exits:  241 (48.2%)
Layer 12 Exits: 36 (7.2%)

[Step 3/3] Ready! Enter a review (type 'exit' to stop).
